# LumiTrack - Machine Learning Analysis Notebook
## Variant 10.03: Museum Exposure Limiter

This notebook provides interactive model exploration, feature importance inspection, polar angle-intensity visualization, and baseline comparison.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src to path
sys.path.append(os.path.abspath('../src'))
from feature_extraction import APPROVED_ANGLES_DEG
from inference import LumiTrackPredictor

### 1. Load Dataset and Inspect Class Distribution

In [ ]:
data_path = '../data/seed_scan_dataset.csv'
df = pd.read_csv(data_path)
print(f'Total Trials: {len(df)}')
df['regime_name'].value_counts().plot(kind='bar', color=['skyblue', 'salmon', 'khaki'])
plt.title('Distribution of Light Regimes')
plt.ylabel('Trial Count')
plt.show()

### 2. Polar Angle-Intensity Profiles
Visualizing representative angular scans for Normal Diffuse, Overexposure Risk, and Specular Confuser.

In [ ]:
fig = plt.figure(figsize=(15, 5))
angles_rad = np.radians(APPROVED_ANGLES_DEG)
lux_cols = [f'lux_pos{i}_{deg}deg' for i, deg in enumerate(APPROVED_ANGLES_DEG)]

regimes = ['NORMAL_DIFFUSE', 'OVEREXPOSURE_RISK', 'CONFUSER_SPECULAR']
colors = ['blue', 'red', 'orange']

for i, reg in enumerate(regimes):
    sample = df[df['regime_name'] == reg].iloc[0]
    lux_vals = sample[lux_cols].values
    
    ax = fig.add_subplot(1, 3, i + 1, polar=True)
    ax.plot(angles_rad, lux_vals, marker='o', color=colors[i], linewidth=2)
    ax.fill(angles_rad, lux_vals, color=colors[i], alpha=0.25)
    ax.set_theta_zero_location('N')
    ax.set_title(f'{reg}\nMax: {np.max(lux_vals):.1f} Lux', va='bottom')

plt.tight_layout()
plt.show()

### 3. Model Benchmark Summary

In [ ]:
with open('../models/metrics_summary.json', 'r') as f:
    metrics = json.load(f)

perf_df = pd.DataFrame(metrics['comparison_table'])
display(perf_df[['model_name', 'accuracy', 'macro_f1', 'overexposure_recall', 'overexposure_false_alarm']])

### 4. Interactive Live Prediction and Confidence-Aware Partial Stopping

In [ ]:
predictor = LumiTrackPredictor()

# Simulate a hazardous direct solar beam entering from 15 deg (POS_0)
sim_hazard_scan = [1250.0, 680.0, 180.0, 85.0, 60.0, 50.0, 45.0]

# Check partial scan after POS_0 to POS_3
partial_res = predictor.evaluate_partial_scan(sim_hazard_scan[:4], target_lux=400.0)
print('Partial Scan Evaluation (POS_0 to POS_3):')
print(f'Early Stop Triggered: {partial_res["early_stop"]}')
print(f'Predicted Regime: {partial_res["regime_name"]} (Confidence: {partial_res["confidence"]})')

# Full scan evaluation
full_res = predictor.evaluate_full_scan(sim_hazard_scan, target_lux=400.0)
print('\nFull Scan Evaluation:')
print(f'Predicted Regime: {full_res["regime_name"]} (Confidence: {full_res["confidence"]})')
print(f'Command Dispatched: {full_res["command"]} -> Angle ID {full_res["recommended_angle_id"]} ({full_res["recommended_angle_deg"]}°)')